# 국내 산림 탄소 저장량 파이프라인
경계와 임상도를 바꾸어 전국 어디든 적용함. 셀을 위에서부터 순서대로 실행함. 기존 결과 폴더는 덮어쓰지 않음.

## 1. Drive와 실행 경로를 지정함
이 셀의 두 경로를 본인 Drive 폴더에 맞춤. EXPERIMENT_DIR에 이 실험 폴더 전체를 먼저 복사함.

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/글루리'
EXPERIMENT_DIR = '/content/drive/MyDrive/글루리/github/geumjeongsan-carbon-poc/experiments/20261113_kr_forest_carbon_stock'
# 재실행할 때에는 새 이름으로 바꿈
RUN_LABEL = 'run01'
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys
os.environ['KFCS_DRIVE_ROOT'] = DRIVE_ROOT
os.chdir(EXPERIMENT_DIR)
sys.path.insert(0, EXPERIMENT_DIR)
OUTPUT_ROOT = Path(DRIVE_ROOT) / '02_분석결과/20261113_국가기준저장량파이프라인_v01' / RUN_LABEL
print(EXPERIMENT_DIR, OUTPUT_ROOT)


## 2. 필요한 패키지를 설치함
SHP, GeoTIFF, XLS와 XLSX를 읽는 패키지를 함께 설치함.

In [ ]:
%pip -q install geopandas pyogrio shapely pandas numpy rasterio xlrd openpyxl pyyaml matplotlib pytest


## 3. 경로를 확인하고 원본 재적표에서 공통 CSV를 생성함
원본 ZIP은 CP949 파일명을 복구하여 메모리에서 읽음. 원본은 변경하지 않음. 기존 캐시는 원본 해시를 검증함.

In [ ]:
from kfcs.config import load_config
from kfcs.volume.tables import ensure_table
c = load_config('configs/geumjeongsan_2025.yaml')
for p in c['inputs']['forest_maps'] + [c['inputs']['sfs_zip'], c['inputs']['kosis_xlsx'], c['boundary']['path']]:
    assert Path(p).exists(), p
for table in ['kosis2020','sfs2025']:
    print(ensure_table(c, table))


## 4. 가짜 재적 래스터로 공급자를 검증함
부분 픽셀, nodata, 픽셀당 재적, 서로 다른 CRS를 시험함.

In [ ]:
import subprocess
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests'], check=True)


## 5. 4주차 방식 재현을 실행함. 기준 대비 1% 초과 차이도 중단 없이 보고함.

In [ ]:
from kfcs.run import run
from IPython.display import display, Markdown, Image
c = load_config('configs/geumjeongsan_week4_repro.yaml', output_dir=OUTPUT_ROOT / 'geumjeongsan_week4_repro')
result = run(c)
display(Markdown((Path(c['output_dir']) / 'report.md').read_text(encoding='utf-8')))
display(Image(filename=str(Path(c['output_dir']) / 'carbon_density_map.png')))


## 6. 2025 국가통계 방식과 몬테카를로를 실행함. 곰솔 불확도 덮어쓰기 적용·미적용 결과도 같은 자료로 비교함. 재적표·1영급 규칙·CF를 순서대로 바꾸고 임상 3종 계수도 별도로 비교함.

In [ ]:
from kfcs.run import run
from IPython.display import display, Markdown, Image
c = load_config('configs/geumjeongsan_2025.yaml', output_dir=OUTPUT_ROOT / 'geumjeongsan_2025')
result = run(c)
display(Markdown((Path(c['output_dir']) / 'report.md').read_text(encoding='utf-8')))
display(Image(filename=str(Path(c['output_dir']) / 'carbon_density_map.png')))


## 7. 금정구 전체 임상도를 행정구역별 공식 통계와 검산함. 면적·재적의 차이율과 원인 후보를 보고함.

In [ ]:
from kfcs.run import run
from IPython.display import display, Markdown, Image
c = load_config('configs/geumjeong_gu_check.yaml', output_dir=OUTPUT_ROOT / 'geumjeong_gu_check')
result = run(c)
display(Markdown((Path(c['output_dir']) / 'report.md').read_text(encoding='utf-8')))
display(Image(filename=str(Path(c['output_dir']) / 'carbon_density_map.png')))


## 8. 국가 인벤토리 방식으로 계산함
NID 2025 임상별 계수와 혼효림 재적 50:50 배분을 적용함. 계수별 불확도는 자료가 없어 고정함. 계수 세트만·혼효림 규칙만·둘 다 바꾸는 네 조합을 보고함.

In [ ]:
c = load_config('configs/geumjeongsan_national_inventory.yaml', output_dir=OUTPUT_ROOT / 'geumjeongsan_national_inventory')
result = run(c)
display(Markdown((Path(c['output_dir']) / 'report.md').read_text(encoding='utf-8')))
display(Image(filename=str(Path(c['output_dir']) / 'carbon_density_map.png')))


## 9. 결과 위치를 확인함
큰 결과는 Drive에 보관함. 저장소에는 1MB 미만 요약과 경로를 상대화한 provenance 사본만 공유함.

In [ ]:
print(OUTPUT_ROOT)
for p in sorted(OUTPUT_ROOT.glob('*/report.md')):
    print(p)

# 작은 공유 사본만 새 실행 이름 아래에 저장함
from kfcs.publish import publish_summaries
for folder in sorted(OUTPUT_ROOT.iterdir()):
    if folder.is_dir() and (folder / 'provenance.json').exists():
        print(folder.name, publish_summaries(folder, Path(EXPERIMENT_DIR) / 'outputs' / RUN_LABEL / folder.name, EXPERIMENT_DIR, DRIVE_ROOT))
